# Football AI · Soccer Scouting (modelos do roboflow/sports)

Este notebook segue o código do tutorial oficial da Roboflow (*football-ai.ipynb*): os modelos YOLOv8 do repositório oficial `roboflow/sports` (jogadores e linhas do campo), rodando direto na GPU do Colab com PyTorch, a mesma separação dos times (SigLIP + agrupamento) e o mesmo radar. O que ele tem a mais: roda no **seu vídeo**, gera o vídeo inteiro marcado e o arquivo para importar na aba **Tático** do Soccer Scouting.

**Antes de começar (uma vez só):**
1. **GPU:** menu **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU** → Salvar.
2. Não precisa de chave da Roboflow: os modelos são baixados uma vez e ficam guardados no seu Drive.

Depois é só rodar as células na ordem (*Ambiente de execução → Executar tudo*).

**Versão 2:** usa o fork `rodrigodezordi/sports` (calibração e finalizações estão na biblioteca, em `sports/common`). calibração do campo que ignora quadros com só a linha do meio visível e reaproveita a última boa; menos troca de ID no rastreamento; time de cada ID por voto da maioria; **finalizações ao gol** (seção 6b), com mapa, recortes de vídeo e o campo `resumo.finalizacoes` no JSON.

**Mais rápido e retomável:** o vídeo e os resultados ficam no Google Drive; a análise é feita em blocos de 1 minuto que ficam salvos, então se a conexão cair é só rodar de novo e ela continua de onde parou. Os ajustes de velocidade estão na célula 3 (`PULAR_QUADROS`, `CALIBRAR_A_CADA`, `CLASSIFICAR_ID_VEZES`, `GERAR_VIDEO`).

In [ ]:
# 1. Instala as ferramentas (uns 2 minutos). Rode sempre que o Colab abrir uma máquina nova.
import shutil, subprocess, sys, os
TEM_GPU = shutil.which("nvidia-smi") is not None
print("GPU disponível:", TEM_GPU)
if not TEM_GPU:
    print("⚠️ SEM GPU: Ambiente de execução > Alterar o tipo de ambiente de execução > T4 GPU, e rode tudo de novo.")

def instalar(*pacotes):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pacotes], capture_output=True, text=True)
    print(("✅ " if r.returncode == 0 else "❌ ") + " ".join(pacotes))
    if r.returncode != 0: print("\n".join((r.stdout + r.stderr).strip().splitlines()[-8:]))
    return r.returncode == 0

def importa(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

# os modelos rodam com PyTorch (já vem no Colab, com GPU) pelo ultralytics: sem inference/pycuda/onnx
instalar("ultralytics", "supervision", "umap-learn", "gdown", "git+https://github.com/rodrigodezordi/sports.git")
# Pillow íntegra (evita o erro "cannot import name '_Ink'")
import importlib.metadata as md
instalar("--force-reinstall", "--no-deps", f"pillow=={md.version('pillow')}")

for m in ["ultralytics", "supervision", "sports", "umap", "gdown"]:
    print(f"{m:12s}", "ok" if importa(m) else "NÃO INSTALOU")
import torch
print("GPU ligada:", torch.cuda.is_available())
print("Se todos estão 'ok': Ambiente de execução > Reiniciar sessão, e rode da célula 2 em diante.")

In [ ]:
# 2. O vídeo. Se ele ainda não estiver no Drive, aparece um botão "Escolher arquivos" para enviar
#    do seu computador. O vídeo é copiado para o Drive (MyDrive/football-ai/videos), e assim, se a
#    conexão cair, não precisa enviar de novo e o progresso da análise continua salvo.
import os, shutil
from google.colab import files, drive
drive.mount("/content/drive")
PASTA_VIDEOS = "/content/drive/MyDrive/football-ai/videos"
PASTA_BASE = "/content/drive/MyDrive/football-ai/resultados"
os.makedirs(PASTA_VIDEOS, exist_ok=True)

NOME_VIDEO = ""   # deixe vazio para enviar do computador; ou escreva o nome de um vídeo que já está no Drive, ex.: "jogo.mp4"
no_drive = sorted(f for f in os.listdir(PASTA_VIDEOS) if f.lower().endswith((".mp4", ".mov", ".avi", ".mkv")))
if NOME_VIDEO and NOME_VIDEO in no_drive:
    VIDEO = os.path.join(PASTA_VIDEOS, NOME_VIDEO)
else:
    if no_drive: print("Vídeos que já estão no Drive:", ", ".join(no_drive), "(para reusar, escreva o nome em NOME_VIDEO)")
    print("Escolha o vídeo no seu computador:")
    enviado = files.upload()
    nome = list(enviado)[0]
    VIDEO = os.path.join(PASTA_VIDEOS, nome)
    shutil.move(nome, VIDEO)   # guarda no Drive
    print(f'Guardado no Drive. Se precisar rodar de novo (ex.: a conexão caiu), escreva NOME_VIDEO = "{nome}" para não enviar outra vez.')
print("Vídeo:", VIDEO)

In [ ]:
# 3. Ajustes (pode deixar como está)
NOME_JOGO = "Meu jogo"     # aparece no app e dá nome à pasta de resultados
INICIO_SEG = 0             # analisar a partir deste segundo do vídeo
FIM_SEG = None             # até este segundo (None = até o fim)
DADOS_POR_SEG = 5          # quantas posições por segundo vão para o arquivo do app
CONFIANCA = 0.3            # a mesma do tutorial oficial
RADAR_TAMANHO = 0.33       # largura do radar em relação ao vídeo (0.33 = um terço)
RADAR_CANTO = "direita"    # "direita" ou "esquerda" (canto de cima)
IGNORAR_FORA_DO_CAMPO = True  # não marca banco, torcida e quem está fora das linhas
MARGEM_M = 0.5             # folga, em metros, além da linha do campo

# --- Velocidade (os valores abaixo deixam a análise ~4x mais rápida que antes) ---
PULAR_QUADROS = 2          # analisa 1 a cada 2 quadros (30 fps -> 15 fps). 1 = todos; 3 = ainda mais rápido
CALIBRAR_A_CADA = 5        # recalcula as linhas do campo a cada 5 quadros analisados (a câmera mexe devagar)
CLASSIFICAR_ID_VEZES = 5   # olha o uniforme de cada jogador só nas 5 primeiras vezes; depois o time fica fixo
GERAR_VIDEO = True         # False = só os dados (bem mais rápido; sem o vídeo marcado)

# --- Retomar se a conexão cair ---
BLOCO_SEG = 60             # salva o progresso a cada 60 s de vídeo analisado

PASTA = os.path.join(PASTA_BASE, "".join(ch if ch.isalnum() else "-" for ch in NOME_JOGO.lower()).strip("-"))
os.makedirs(os.path.join(PASTA, "blocos"), exist_ok=True)
print("Resultados em:", PASTA)

In [ ]:
# 4. Carrega os modelos (na GPU) e aprende os uniformes dos dois times (como no tutorial)
import torch
import cv2, json, numpy as np, supervision as sv
from tqdm.auto import tqdm
import subprocess
from ultralytics import YOLO
from sports.common.team import TeamClassifier
from sports.common.view import ViewTransformer
from sports.common.calibration import PitchCalibrator
from sports.common.shots import detect_shots, ShotDetectorConfig
from sports.configs.soccer import SoccerPitchConfiguration
from sports.annotators.soccer import draw_pitch, draw_points_on_pitch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
if DEV == "cpu": print("ATENÇÃO: sem GPU fica muito lento. Ambiente de execução > Alterar o tipo de ambiente > T4 GPU, e rode tudo de novo.")
# modelos YOLOv8 do roboflow/sports (os mesmos do examples/soccer/setup.sh), guardados no Drive
PESOS = os.path.join(os.path.dirname(PASTA_BASE), "modelos"); os.makedirs(PESOS, exist_ok=True)
LINKS = {"football-player-detection.pt": "17PXFNlx-jI7VjVo_vQnB1sONjRyvoB-q",
         "football-pitch-detection.pt": "1Ma5Kt86tgpdjCTKfum79YMgNnSjcoOyf"}
for nome_pt, gid in LINKS.items():
    destino = os.path.join(PESOS, nome_pt)
    if not os.path.exists(destino):
        subprocess.run(["gdown", "-q", "-O", destino, f"https://drive.google.com/uc?id={gid}"], check=True)
PLAYER_DETECTION_MODEL = YOLO(os.path.join(PESOS, "football-player-detection.pt"))
FIELD_DETECTION_MODEL = YOLO(os.path.join(PESOS, "football-pitch-detection.pt"))
HALF = DEV == "cuda"   # meia precisão: mais rápido na GPU
print("Modelos rodando em:", DEV.upper())
BALL_ID, GOALKEEPER_ID, PLAYER_ID, REFEREE_ID = 0, 1, 2, 3
CONFIG = SoccerPitchConfiguration()
CORES = ["#00BFFF", "#FF1493", "#FFD700"]  # as cores do tutorial: time 0, time 1, árbitro

info = sv.VideoInfo.from_video_path(VIDEO)
FPS = info.fps
ini_f = int(INICIO_SEG * FPS)
fim_f = int(FIM_SEG * FPS) if FIM_SEG else info.total_frames
def quadros(stride=1):
    return sv.get_video_frames_generator(VIDEO, start=ini_f, end=fim_f, stride=stride)

def detectar(frame):
    return sv.Detections.from_ultralytics(PLAYER_DETECTION_MODEL(frame, imgsz=1280, conf=CONFIANCA, device=DEV, half=HALF, verbose=False)[0])

# calibração do campo (biblioteca sports): ignora quadros com pontos alinhados
# (ex.: só a linha do meio) e reaproveita a última calibração boa por até 1 s
calibrador = PitchCalibrator(CONFIG, confidence_threshold=0.5, max_age=int(FPS))
def keypoints(frame):
    return sv.KeyPoints.from_ultralytics(FIELD_DETECTION_MODEL(frame, device=DEV, half=HALF, verbose=False)[0])
def transformador(frame, cache=True):
    kp = keypoints(frame)
    return calibrador.update(kp) if cache else calibrador.compute(kp)

def no_campo(det, vt):
    if not IGNORAR_FORA_DO_CAMPO or vt is None or not len(det): return np.ones(len(det), bool)
    xy = vt.transform_points(det.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)) / 100
    return (xy[:, 0] >= -MARGEM_M) & (xy[:, 0] <= 120 + MARGEM_M) & (xy[:, 1] >= -MARGEM_M) & (xy[:, 1] <= 70 + MARGEM_M)

def resolve_goalkeepers_team_id(players, goalkeepers):
    # do tutorial: o goleiro é do time cujos jogadores estão, em média, mais perto dele
    if not len(goalkeepers): return np.array([], int)
    if len(set(players.class_id.tolist())) < 2: return np.zeros(len(goalkeepers), int)
    goalkeepers_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    team_0_centroid = players_xy[players.class_id == 0].mean(axis=0)
    team_1_centroid = players_xy[players.class_id == 1].mean(axis=0)
    return np.array([0 if np.linalg.norm(g - team_0_centroid) < np.linalg.norm(g - team_1_centroid) else 1 for g in goalkeepers_xy])

# uniformes dos dois times: aprendidos uma vez e salvos no Drive. Ao retomar, usa os mesmos
# (senão o "time 0" de agora poderia ser o "time 1" de antes)
import pickle
team_classifier = TeamClassifier(device=DEV, verbose=False)
arq_times = os.path.join(PASTA, "times.pkl")
if os.path.exists(arq_times):
    with open(arq_times, "rb") as fh: team_classifier.reducer, team_classifier.cluster_model = pickle.load(fh)
    print("Uniformes carregados de", arq_times)
else:
    # recortes dos jogadores, 1 quadro a cada 30 (como no tutorial)
    crops = []
    for frame in tqdm(quadros(stride=30), desc="aprendendo os uniformes"):
        d = detectar(frame); d = d[d.class_id == PLAYER_ID]
        d = d[no_campo(d, transformador(frame, cache=False))]
        crops += [sv.crop_image(frame, xyxy) for xyxy in d.xyxy]
    team_classifier.fit(crops)
    with open(arq_times, "wb") as fh: pickle.dump((team_classifier.reducer, team_classifier.cluster_model), fh)
    print("Recortes usados:", len(crops), "· jogadores em cada time:", np.bincount(team_classifier.predict(crops), minlength=2))

In [ ]:
# 5. Analisa o vídeo em blocos de BLOCO_SEG segundos. Cada bloco pronto fica salvo em PASTA/blocos.
#    Se a conexão cair: rode de novo as células 1 a 4 e esta. Os blocos já feitos são pulados.
import glob, subprocess
ellipse_annotator = sv.EllipseAnnotator(color=sv.ColorPalette.from_hex(CORES), thickness=2)
label_annotator = sv.LabelAnnotator(color=sv.ColorPalette.from_hex(CORES), text_color=sv.Color.from_hex("#000000"), text_position=sv.Position.BOTTOM_CENTER)
triangle_annotator = sv.TriangleAnnotator(color=sv.Color.from_hex("#FFD700"), base=20, height=17)

FPS_ANALISE = FPS / PULAR_QUADROS
# memória de 3 s: o jogador que some atrás de outro volta com o mesmo ID
tracker = sv.ByteTrack(frame_rate=max(1, int(FPS_ANALISE)), lost_track_buffer=int(3 * FPS_ANALISE), minimum_consecutive_frames=3)
tracker.reset()
# a calibração só é recalculada a cada CALIBRAR_A_CADA quadros; a última boa vale por até ~1 s
calibrador = PitchCalibrator(CONFIG, confidence_threshold=0.5,
                             max_age=max(1, round(FPS_ANALISE / CALIBRAR_A_CADA)))
import collections
votos_time = collections.defaultdict(collections.Counter)   # ID -> votos de time (cache do uniforme)
passo_dados = max(1, round(FPS_ANALISE / DADOS_POR_SEG))
info_saida = sv.VideoInfo(width=info.width, height=info.height, fps=FPS_ANALISE)

def time_dos_jogadores(frame, players):
    # classifica pelo uniforme só os IDs com poucos votos; os outros usam o time já decidido
    if not len(players): return np.array([], int)
    novos = [k for k, tid in enumerate(players.tracker_id) if sum(votos_time[tid].values()) < CLASSIFICAR_ID_VEZES]
    if novos:
        pred = team_classifier.predict([sv.crop_image(frame, players.xyxy[k]) for k in novos])
        for k, t in zip(novos, pred): votos_time[players.tracker_id[k]][int(t)] += 1
    return np.array([votos_time[tid].most_common(1)[0][0] for tid in players.tracker_id], int)

def desenhar(frame, all_detections, ball_detections, xy_campo, bola_campo):
    out = frame.copy()
    if len(all_detections):
        labels = [f"#{tracker_id}" for tracker_id in all_detections.tracker_id]
        out = ellipse_annotator.annotate(scene=out, detections=all_detections)
        out = label_annotator.annotate(scene=out, detections=all_detections, labels=labels)
    if len(ball_detections): out = triangle_annotator.annotate(scene=out, detections=ball_detections)
    if xy_campo is None and bola_campo is None: return out
    # radar (visão de cima), como no tutorial, com as 3 linhas das 4 zonas
    radar = draw_pitch(config=CONFIG, padding=50, scale=0.1)
    if bola_campo is not None:
        radar = draw_points_on_pitch(config=CONFIG, xy=np.array([bola_campo]), face_color=sv.Color.WHITE, edge_color=sv.Color.BLACK, radius=10, pitch=radar)
    if xy_campo is not None:
        for k in (0, 1, 2):
            pts = xy_campo[all_detections.class_id == k]
            if len(pts): radar = draw_points_on_pitch(config=CONFIG, xy=pts, face_color=sv.Color.from_hex(CORES[k]), edge_color=sv.Color.BLACK, radius=16, pitch=radar)
    for zx in (3000, 6000, 9000):
        px = int(50 + zx * 0.1)
        for y0 in range(50, 50 + 700, 24): cv2.line(radar, (px, y0), (px, min(y0 + 14, 750)), (79, 213, 255), 3)
    h, w = out.shape[:2]; radar = sv.resize_image(radar, (int(w * RADAR_TAMANHO), h), keep_aspect_ratio=True); rh, rw = radar.shape[:2]
    rx = 10 if RADAR_CANTO == "esquerda" else w - rw - 10
    return sv.draw_image(out, radar, opacity=0.85, rect=sv.Rect(x=rx, y=10, width=rw, height=rh))

def analisar_bloco(b0, b1, id_extra, caminho_video):
    dados_b, bola_b = [], []
    sink = sv.VideoSink(caminho_video, info_saida) if GERAR_VIDEO else None
    if sink: sink.__enter__()
    vt = None
    try:
        gen = sv.get_video_frames_generator(VIDEO, start=b0, end=b1, stride=PULAR_QUADROS)
        for j, frame in enumerate(tqdm(gen, total=(b1 - b0 + PULAR_QUADROS - 1) // PULAR_QUADROS, leave=False, desc=f"{b0 / FPS:.0f}-{b1 / FPS:.0f}s")):
            f_idx = b0 + j * PULAR_QUADROS
            if j % CALIBRAR_A_CADA == 0: vt = transformador(frame)
            detections = detectar(frame)

            ball_detections = detections[detections.class_id == BALL_ID]
            ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)
            all_detections = detections[detections.class_id != BALL_ID]
            all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
            all_detections = all_detections[no_campo(all_detections, vt)]
            all_detections = tracker.update_with_detections(detections=all_detections)
            all_detections.tracker_id = all_detections.tracker_id + id_extra  # IDs únicos mesmo depois de retomar

            goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
            players_detections = all_detections[all_detections.class_id == PLAYER_ID]
            referees_detections = all_detections[all_detections.class_id == REFEREE_ID]
            players_detections.class_id = time_dos_jogadores(frame, players_detections)
            goalkeepers_detections.class_id = resolve_goalkeepers_team_id(players_detections, goalkeepers_detections)
            referees_detections.class_id -= 1
            n_jog, n_gol = len(players_detections), len(goalkeepers_detections)
            all_detections = sv.Detections.merge([players_detections, goalkeepers_detections, referees_detections])
            all_detections.class_id = all_detections.class_id.astype(int)

            xy_campo, bola_campo = None, None
            if vt is not None:
                if len(all_detections): xy_campo = vt.transform_points(all_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER))
                if len(ball_detections):  # bola de maior confiança
                    bola_campo = vt.transform_points(ball_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER))[int(np.argmax(ball_detections.confidence))]
            if sink: sink.write_frame(desenhar(frame, all_detections, ball_detections, xy_campo, bola_campo))

            if bola_campo is not None:
                bxy = bola_campo / 100; dono = (None, None, None)
                if xy_campo is not None:
                    m = np.isin(all_detections.class_id, [0, 1])
                    if m.any():
                        dd = np.hypot(*(xy_campo[m] / 100 - bxy).T); k = int(np.argmin(dd))
                        dono = (int(all_detections.tracker_id[m][k]), int(all_detections.class_id[m][k]), float(dd[k]))
                bola_b.append(dict(frame=f_idx, t=f_idx / FPS, x_m=float(bxy[0]), y_m=float(bxy[1]),
                                   owner_id=dono[0], owner_team=dono[1], owner_dist_m=dono[2]))

            if j % passo_dados == 0:
                p = []
                if xy_campo is not None:
                    gk = [0] * n_jog + [1] * n_gol + [0] * (len(all_detections) - n_jog - n_gol)
                    for tid, t, (x, y), g in zip(all_detections.tracker_id, all_detections.class_id, xy_campo / 100, gk):
                        if -MARGEM_M <= x <= 120 + MARGEM_M and -MARGEM_M <= y <= 70 + MARGEM_M:
                            p.append([int(tid), 3 if int(t) == 2 else int(t), round(float(x), 1), round(float(y), 1), g])  # no app, árbitro = 3
                b = None if bola_campo is None else [round(float(bola_campo[0]) / 100, 1), round(float(bola_campo[1]) / 100, 1)]
                dados_b.append({"t": round(f_idx / FPS, 2), "p": p, "b": b})
    finally:
        if sink: sink.__exit__(None, None, None)
    return dados_b, bola_b

tam_bloco = int(BLOCO_SEG * FPS)
blocos = [(b0, min(b0 + tam_bloco, fim_f)) for b0 in range(ini_f, fim_f, tam_bloco)]
id_extra = None   # definido no primeiro bloco analisado nesta sessão
for n, (b0, b1) in enumerate(tqdm(blocos, desc="blocos")):
    arq = os.path.join(PASTA, "blocos", f"bloco_{n:03d}.json")
    if os.path.exists(arq): continue   # já feito numa execução anterior
    if id_extra is None: id_extra = n * 100000   # IDs de uma sessão nova não se misturam com os antigos
    dados_b, bola_b = analisar_bloco(b0, b1, id_extra, os.path.join(PASTA, "blocos", f"bloco_{n:03d}.mp4"))
    with open(arq + ".tmp", "w") as fh: json.dump({"dados": dados_b, "bola": bola_b}, fh)
    os.replace(arq + ".tmp", arq)   # só conta como feito quando terminou de gravar

# junta todos os blocos
dados, bola_log = [], []
for n in range(len(blocos)):
    with open(os.path.join(PASTA, "blocos", f"bloco_{n:03d}.json")) as fh: d = json.load(fh)
    dados += d["dados"]; bola_log += d["bola"]
SAIDA = os.path.join(PASTA, "jogo_marcado.mp4")
partes = [os.path.join(PASTA, "blocos", f"bloco_{n:03d}.mp4") for n in range(len(blocos))]
if GERAR_VIDEO and all(os.path.exists(x) for x in partes):
    lista = os.path.join(PASTA, "blocos", "lista.txt")
    with open(lista, "w") as fh: fh.writelines(f"file '{x}'\n" for x in partes)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", lista, "-c", "copy", SAIDA], check=True)
print("Pronto:", len(dados), "registros de posição ·", len(bola_log), "quadros com bola")

In [ ]:
# 6. Resumo: posse de bola, distância percorrida e posição média
import collections, math
# time de cada ID = o que ele recebeu na maioria dos quadros (corrige "piscadas" do classificador)
votos = collections.defaultdict(collections.Counter)
for q in dados:
    for tid, t, *_ in q["p"]:
        if t in (0, 1): votos[tid][t] += 1
TIME_ID = {tid: c.most_common(1)[0][0] for tid, c in votos.items()}
for q in dados:
    for r in q["p"]:
        if r[1] in (0, 1): r[1] = TIME_ID[r[0]]
posse = collections.Counter(); dist = collections.defaultdict(float); ult = {}; time_de = {}
soma = {0: [0, 0, 0], 1: [0, 0, 0]}
for q in dados:
    if q["b"] is not None and q["p"]:
        bx, by = q["b"]; perto = min((math.hypot(x - bx, y - by), t) for _, t, x, y, _ in q["p"] if t in (0, 1)) if any(t in (0, 1) for _, t, _, _, _ in q["p"]) else (99, None)
        if perto[0] < 2.5: posse[perto[1]] += 1
    for tid, t, x, y, g in q["p"]:
        if t not in (0, 1): continue
        time_de[tid] = t; soma[t][0] += x; soma[t][1] += y; soma[t][2] += 1
        if tid in ult:
            d = math.hypot(x - ult[tid][0], y - ult[tid][1]); dt = q["t"] - ult[tid][2]
            if 0 < dt < 1.5 and d / dt < 11: dist[tid] += d   # ignora saltos (troca de ID)
        ult[tid] = (x, y, q["t"])
tp = posse[0] + posse[1]
resumo = {"posse": [round(100 * posse[0] / tp) if tp else None, round(100 * posse[1] / tp) if tp else None],
          "posicaoMedia": [[round(soma[k][0] / soma[k][2], 1), round(soma[k][1] / soma[k][2], 1)] if soma[k][2] else None for k in (0, 1)],
          "distancia": {str(k): round(v) for k, v in sorted(dist.items(), key=lambda kv: -kv[1])[:40]},
          "timeDoId": {str(k): v for k, v in time_de.items()}}
print("Posse de bola (time 0 / time 1):", resumo["posse"], "%")
print("Posição média (x de 0 a 120 m, y de 0 a 70 m):", resumo["posicaoMedia"])
print("Quem mais correu (ID: metros):", list(resumo["distancia"].items())[:10])

## 6b. Finalizações ao gol

Ideia do demo de basquete (`jeremyipark/vision-demos`): um arremesso é um evento com regras geométricas da bola, e cada um vira um ponto no mapa. Aqui, um **chute** é contado quando, em sequência:
1. **Posse:** a bola estava a até 2 m de um jogador no último 1 s (ele é o finalizador).
2. **Velocidade:** a bola sai a pelo menos 12 m/s (~43 km/h).
3. **Zona:** parte a até 35 m do centro do gol.
4. **Direção:** a trajetória prolongada cruza a linha de fundo na boca do gol, com até 4 m de folga. Passe forte e inversão ficam de fora.
5. **Um chute por lance** (2 s de intervalo).

"Na direção do gol" = a linha passa entre as traves (7,32 m); senão "para fora". **Gol ou não, a câmera não permite afirmar** (a bola sai do chão): confira pelos recortes de vídeo salvos de cada chute. Limites em `SHOT_PARAMS` (`ShotDetectorConfig`, em `sports/common/shots.py`).

In [ ]:
import pandas as pd
L_M, W_M = CONFIG.length / 100, CONFIG.width / 100
SHOT_PARAMS = ShotDetectorConfig()   # limites ajustáveis, ex.: ShotDetectorConfig(min_speed_ms=10)
RESULTADO = {"on_target": "na direção do gol", "off_target": "para fora"}

chutes = detect_shots(bola_log, FPS, L_M, W_M, team_of=TIME_ID, config=SHOT_PARAMS)
chutes = chutes.rename(columns={"player_id": "id", "goal": "gol", "team": "time", "distance_m": "distancia_m",
                                "angle_deg": "angulo_graus", "ball_speed_kmh": "vel_bola_kmh",
                                "outcome": "resultado"})
chutes["resultado"] = chutes["resultado"].map(RESULTADO)
chutes["gol"] = chutes["gol"].map({"left": "esquerdo", "right": "direito"})
print(f"{len(chutes)} finalizações · bola vista em {len(bola_log)} de {fim_f - ini_f} quadros")
if len(chutes):
    print(chutes.groupby("time").agg(chutes=("frame", "size"),
          na_direcao=("resultado", lambda s: (s == "na direção do gol").sum())).to_string())
resumo["finalizacoes"] = chutes.to_dict("records")
chutes

In [ ]:
# Mapa de finalizações + recorte de 3 s de cada chute para conferir
import matplotlib.pyplot as plt
if len(chutes):
    fig = plt.figure(figsize=(11, 6.5)); ax = fig.gca()
    v = np.array(CONFIG.vertices) / 100
    for a, b_ in CONFIG.edges: ax.plot(*v[[a - 1, b_ - 1]].T, color="white", lw=1.2)
    ax.add_patch(plt.Circle((L_M / 2, W_M / 2), CONFIG.centre_circle_radius / 100, fill=False, color="white"))
    ax.set_facecolor("#2e7d32"); ax.set_xlim(-2, L_M + 2); ax.set_ylim(W_M + 2, -2); ax.set_aspect("equal")
    for tm in (0, 1):
        for res, mk in [("na direção do gol", "o"), ("para fora", "X")]:
            s_ = chutes[(chutes.time == tm) & (chutes.resultado == res)]
            ax.scatter(s_.x, s_.y, s=90 + 3 * s_.vel_bola_kmh, c=CORES[tm], marker=mk, edgecolors="black", zorder=5,
                       label=f"Time {tm}: {res} ({len(s_)})")
            for _, r in s_.iterrows():
                ax.annotate("", xy=(0 if r.gol == "esquerdo" else L_M, W_M / 2), xytext=(r.x, r.y),
                            arrowprops=dict(arrowstyle="->", color=CORES[tm], alpha=0.5))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2); ax.set_title("Finalizações (tamanho = velocidade da bola)")
    plt.savefig(os.path.join(PASTA, "mapa_finalizacoes.png"), dpi=150, bbox_inches="tight"); plt.show()
    for n, r in chutes.reset_index(drop=True).iterrows():
        with sv.VideoSink(os.path.join(PASTA, f"chute_{n + 1:02d}_t{r.t:.0f}s.mp4"), info) as s_:
            for fr in sv.get_video_frames_generator(VIDEO, start=max(0, int(r.frame - FPS)), end=int(r.frame + 2 * FPS)):
                s_.write_frame(fr)
    chutes.to_csv(os.path.join(PASTA, "finalizacoes.csv"), index=False)

In [ ]:
# 7. Salva e baixa: o vídeo marcado e o arquivo para importar no Soccer Scouting
nome = os.path.join(PASTA, "football-ai-" + "".join(ch if ch.isalnum() else "-" for ch in NOME_JOGO.lower()).strip("-") + ".json")
json.dump({"tipo": "football-ai", "versao": 1, "jogo": NOME_JOGO, "video": VIDEO, "fps": DADOS_POR_SEG,
           "inicio": INICIO_SEG, "campo": {"comprimento": 120, "largura": 70},
           "times": [{"id": 0, "cor": CORES[0]}, {"id": 1, "cor": CORES[1]}],
           "frames": dados, "resumo": resumo}, open(nome, "w"), separators=(",", ":"))
files.download(nome)
if GERAR_VIDEO and os.path.exists(SAIDA): files.download(SAIDA)
import glob
for f_ in [os.path.join(PASTA, x) for x in ("finalizacoes.csv", "mapa_finalizacoes.png")] + sorted(glob.glob(os.path.join(PASTA, "chute_*.mp4"))):
    if os.path.exists(f_): files.download(f_)
print("Tudo também ficou salvo em", PASTA)
print("Importe", os.path.basename(nome), "na aba Tático do Soccer Scouting (botão Importar análise do Colab).")